# Question 3: Fire Station Data Analysis for Social Impact

**Overview**
- Using redacted version of real data about a firefighting station in Midlands South Carolina 2025
- Solving Data Issues
- Exploratory Data Analysis
- Unsupervised Learning
- 2,200 emergency incidents, 8 columns of data
- 160 Days of emergencys

**Analysis**
1. Find Range of data for cases (dispatches)
2. Find percent of data missing by each column
3. Find and solve data issues
4. On average, how much time is a call (alarm) from open to close
5. How many fire units, on average, are sent for a fire alarm
6. Among A, B, and C, which shift is busiest
7. Create matix of fire alarms by day and week and hour of the day
8. Cluster data by Anges and K-value to report quality of cluster
9. interpret clusters and representation

**Section 1:**
- Load Libraries
- examine fire station data

In [12]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from datetime import datetime
from sklearn.preprocessing import StandardScaler
from sklearn.cluster import KMeans, AgglomerativeClustering as AGNES
from sklearn.metrics import silhouette_score

df = pd.read_csv('RedactedFireStation.txt')
print(f"data: {df.shape}")
print(f"Columns: {list(df.columns)}")

data: (2200, 8)
Columns: ['XREF ID', 'DISPATCH UNIT', 'DISPATCH CREATED DATE', 'INCIDENT NUMBER', '1ST UNIT ON SCENE', 'ALARM DATE TIME', 'CALL COMPLETE', 'SHIFT']


**Section 2: Data Quality and Range**
- Finding range of data for cases

In [13]:
date_columns = ['DISPATCH CREATED DATE', 'ALARM DATE TIME', 'CALL COMPLETE']

for col in date_columns:
    if col in df.columns:
        df[col] = pd.to_datetime(df[col], format='%m/%d/%y %H:%M', errors='coerce')
        
if 'ALARM DATE TIME' in df.columns:
    valid_dates = df['ALARM DATE TIME'].dropna()
    if not valid_dates.empty:
        print(f"Date range: {valid_dates.min()} to {valid_dates.max()}")
        print(f"Total span: {(valid_dates.max() - valid_dates.min()).days} days")

Date range: 2025-03-24 15:46:00 to 2025-08-31 22:58:00
Total span: 160 days


**Section 3: Missing Data**
- Finding percentage of missing data 

In [14]:
# Missing data %
missing_percent = (df.isnull().sum() / len(df)) * 100
missing_df = pd.DataFrame({
    'Column': missing_percent.index,
    'Missing %': missing_percent.values
}).sort_values('Missing %', ascending=False)

print("Missing data by each column:")
for _, row in missing_df.iterrows():
    print(f"{row['Column']:30s}: {row['Missing %']:.2f}%")

Missing data by each column:
1ST UNIT ON SCENE             : 19.45%
SHIFT                         : 3.14%
CALL COMPLETE                 : 1.41%
ALARM DATE TIME               : 1.41%
INCIDENT NUMBER               : 0.00%
DISPATCH CREATED DATE         : 0.00%
DISPATCH UNIT                 : 0.00%
XREF ID                       : 0.00%


**Section 4: Data issues**
- find format issues and complete data clense

In [15]:
# data clense
print("Issues:")
print("1. Date: M/D/YY H:MM format")
print("2. wrong dates")
print("3. Multiple units in dispatch unit")
print("4. Empty values in several columns")

df_clean = df.copy()

# unique ids
df_clean['CASE_ID'] = df_clean['XREF ID'].astype(str)

# fix dates
if 'CALL COMPLETE' in df_clean.columns:
    df_clean['CALL COMPLETE'] = pd.to_datetime(df_clean['CALL COMPLETE'], errors='coerce')
    mask = df_clean['CALL COMPLETE'].dt.year < 2000
    df_clean.loc[mask, 'CALL COMPLETE'] = df_clean.loc[mask, 'CALL COMPLETE'] + pd.DateOffset(years=100)

df_clean['NUM_UNITS'] = df_clean['DISPATCH UNIT'].fillna('').str.count(',') + 1
df_clean.loc[df_clean['DISPATCH UNIT'].isna(), 'NUM_UNITS'] = 0

df_clean['SHIFT'] = df_clean['SHIFT'].fillna('Unknown')

print("\ndata clense complete")
print(f"total rows: {len(df_clean)}")

Issues:
1. Date: M/D/YY H:MM format
2. wrong dates
3. Multiple units in dispatch unit
4. Empty values in several columns

data clense complete
total rows: 2200


**Section 5: Exploratory data analysis**
- time between alarm and call

In [16]:
# average response time
print("analyze time diff between dispatch creation and alarm date time\n")

# time in minutes
df_clean['RESPONSE_MINUTES'] = (
    df_clean['ALARM DATE TIME'] - df_clean['DISPATCH CREATED DATE']
).dt.total_seconds() / 60

valid_response = df_clean['RESPONSE_MINUTES'].dropna()

positive_response = valid_response[valid_response > 0]
if not positive_response.empty:
    print(f"\ndispatch before delays:")
    print(f"Mean delay: {positive_response.mean():.2f} min")

df_clean['ABS_RESPONSE'] = df_clean['RESPONSE_MINUTES'].abs()
abs_valid = df_clean['ABS_RESPONSE'].dropna()
print(f"\nTime difference:")
print(f"  Mean: {abs_valid.mean():.2f} minutes")
print(f"  Median: {abs_valid.median():.2f} minutes")

analyze time diff between dispatch creation and alarm date time


dispatch before delays:
Mean delay: 2.00 min

Time difference:
  Mean: 101.22 minutes
  Median: 4.00 minutes


**Section 6: Firestation unit for the alarms**
- average amount of units dispatched per alarm

In [17]:
# average units dispatched
avg_units = df_clean[df_clean['NUM_UNITS'] > 0]['NUM_UNITS'].mean()
print(f"Average number of units dispatched per alarm: {avg_units:.2f}")

unit_dist = df_clean[df_clean['NUM_UNITS'] > 0]['NUM_UNITS'].value_counts().sort_index()
print("\nUnit distribution:")
for units, count in unit_dist.head(5).items():
    print(f"  {units} unit(s): {count} alarms")

Average number of units dispatched per alarm: 1.44

Unit distribution:
  1 unit(s): 1530 alarms
  2 unit(s): 451 alarms
  3 unit(s): 151 alarms
  4 unit(s): 57 alarms
  5 unit(s): 10 alarms


**Section 7: Fire unit's shift for dispatch**
- average number of units dispatched per each alarm

In [18]:
shift_counts = df_clean['SHIFT'].value_counts()
print("Alarms by shift:")
for shift, count in shift_counts.items():
    if shift in ['A', 'B', 'C']:
        print(f"  Shift {shift}: {count} alarms")

abc_shifts = shift_counts[shift_counts.index.isin(['A', 'B', 'C'])]
if not abc_shifts.empty:
    busiest = abc_shifts.idxmax()
    print(f"\nBusiest shift: {busiest} with {abc_shifts[busiest]} alarms")

Alarms by shift:
  Shift A: 735 alarms
  Shift C: 719 alarms
  Shift B: 677 alarms

Busiest shift: A with 735 alarms


**Section 8: Mattrix**
- day of week by hour of day matrix for fire alarm patterns

In [19]:
# days/hours
df_matrix = df_clean[df_clean['ALARM DATE TIME'].notna()].copy()
df_matrix['DAY_OF_WEEK'] = df_matrix['ALARM DATE TIME'].dt.day_name()
df_matrix['HOUR'] = df_matrix['ALARM DATE TIME'].dt.hour

day_order = ['Monday', 'Tuesday', 'Wednesday', 'Thursday', 'Friday', 'Saturday', 'Sunday']
matrix = pd.crosstab(
    df_matrix['HOUR'],
    df_matrix['DAY_OF_WEEK'],
    margins=True,
    margins_name='TOTAL'
)

# reordering the columns
cols = [day for day in day_order if day in matrix.columns] + ['TOTAL']
matrix = matrix[cols]

print("Fire Alarms by Day of Week and Hour for 10 hours (full 24 hours in fire alarms matrix):")
print(matrix.head(10))

matrix.to_csv('fire_alarms_matrix.csv')

Fire Alarms by Day of Week and Hour for 10 hours (full 24 hours in fire alarms matrix):
DAY_OF_WEEK  Monday  Tuesday  Wednesday  Thursday  Friday  Saturday  Sunday  \
HOUR                                                                          
0                 8        7          5         2       3         7       4   
1                 9       10          7         8       4         7      10   
2                 5        4          3         3       3         8      10   
3                 9        9          9         1      10         5       7   
4                 3        4          5         2       7         6       4   
5                 9        3          6         5       5         3       7   
6                 5        5          7         7      11         7       9   
7                14       15          9        10      14         5      11   
8                13       14         18        10       7         6      14   
9                10       17         14    

**Section 9: Clustering**
- Data for analysis

In [20]:
# clustering
df_ml = df_clean.copy()

df_ml['HOUR'] = df_ml['ALARM DATE TIME'].dt.hour
df_ml['DAY_OF_WEEK'] = df_ml['ALARM DATE TIME'].dt.dayofweek
df_ml['MONTH'] = df_ml['ALARM DATE TIME'].dt.month

feature_cols = ['NUM_UNITS', 'HOUR', 'DAY_OF_WEEK', 'MONTH']

df_cluster = df_ml[feature_cols + ['CASE_ID']].dropna()
X = df_cluster[feature_cols].values

scaler = StandardScaler()
X_scaled = scaler.fit_transform(X)

print(f"Clustering with {len(X_scaled)} samples and {X_scaled.shape[1]} features")

Clustering with 2169 samples and 4 features


**Section 10: Data processing**
- Compares Anges and K-Means to see what is better to use

In [21]:
# methods
n_clusters = 4

# k means
kmeans = KMeans(n_clusters=n_clusters, random_state=42, n_init=10)
kmeans_labels = kmeans.fit_predict(X_scaled)
kmeans_score = silhouette_score(X_scaled, kmeans_labels)

# Agnes
agnes = AGNES(n_clusters=n_clusters)
agnes_labels = agnes.fit_predict(X_scaled)
agnes_score = silhouette_score(X_scaled, agnes_labels)

print("Clustering Results:")
print(f"K-Means Score: {kmeans_score:.4f}")
print(f"Anges Score: {agnes_score:.4f}")

if kmeans_score > agnes_score:
    print("\nk means performs better")
    best_labels = kmeans_labels
else:
    print("\nAGNES performs better")
    best_labels = agnes_labels

Clustering Results:
K-Means Score: 0.2238
Anges Score: 0.1666

k means performs better


**Section 11: comparison and interpretation**

In [22]:
# interpretation
df_cluster['CLUSTER'] = best_labels

cluster_summary = df_cluster.groupby('CLUSTER')[feature_cols].mean().round(2)
cluster_counts = df_cluster['CLUSTER'].value_counts().sort_index()

print("Cluster size:")
for cluster, count in cluster_counts.items():
    print(f"  Cluster {cluster}: {count} incidents")

print("\nCluster mean value:")
print(cluster_summary)

print("interpretation:")

for cluster in range(n_clusters):
    if cluster in cluster_summary.index:
        stats = cluster_summary.loc[cluster]
        
        # on units
        if stats['NUM_UNITS'] > 2:
            size = "Large-scale"
        elif stats['NUM_UNITS'] > 1.5:
            size = "Medium-scale"  
        else:
            size = "Small-scale"
        
        # by hour
        hour = stats['HOUR']
        if 6 <= hour < 12:
            time = "morning"
        elif 12 <= hour < 17:
            time = "afternoon"
        elif 17 <= hour < 22:
            time = "evening"
        else:
            time = "night/early morning"
        
        # by day
        day_val = stats['DAY_OF_WEEK']
        if 0 <= day_val <= 4:
            day_type = "weekday"
        else:
            day_type = "weekend"
            
        print(f"cluster {cluster}: {size} {day_type} incidents in the {time}")
        print(f"  - average {stats['NUM_UNITS']:.1f} units dispatched")
        print(f"  - Peak time: {stats['HOUR']:.0f}:00")

# Save results
df_cluster[['CASE_ID', 'CLUSTER']].to_csv('cluster_assignments.csv', index=False)

Cluster size:
  Cluster 0: 215 incidents
  Cluster 1: 701 incidents
  Cluster 2: 615 incidents
  Cluster 3: 638 incidents

Cluster mean value:
         NUM_UNITS   HOUR  DAY_OF_WEEK  MONTH
CLUSTER                                      
0             3.37  13.43         2.86   5.67
1             1.24  13.39         2.92   7.31
2             1.23  14.37         4.74   4.81
3             1.21  12.22         1.10   4.71
interpretation:
cluster 0: Large-scale weekday incidents in the afternoon
  - average 3.4 units dispatched
  - Peak time: 13:00
cluster 1: Small-scale weekday incidents in the afternoon
  - average 1.2 units dispatched
  - Peak time: 13:00
cluster 2: Small-scale weekend incidents in the afternoon
  - average 1.2 units dispatched
  - Peak time: 14:00
cluster 3: Small-scale weekday incidents in the afternoon
  - average 1.2 units dispatched
  - Peak time: 12:00
